# Train CLMBR

This tutorial walks through the various steps to train a CLMBR model.

Training CLMBR is a three step process:

- Training a tokenizer
- Preparing batches
- Training the model

In [ ]:
import shutil
import os

TARGET_DIR = 'trash/tutorial_4'

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

os.mkdir(TARGET_DIR)

In [ ]:
import datasets
import femr.index
import femr.splits

# First, we want to split our dataset into train, valid, and test
# We do this by calling our split functionality twice

dataset = datasets.Dataset.from_parquet('input/meds/data/*')


index = femr.index.PatientIndex(dataset, num_proc=4)
main_split = femr.splits.generate_hash_split(index.get_patient_ids(), 97, frac_test=0.15)


os.mkdir(os.path.join(TARGET_DIR, 'clmbr_model'))
# Note that we want to save this to the target directory since this is important information

main_split.save_to_csv(os.path.join(TARGET_DIR, "clmbr_model", "main_split.csv"))

train_split = femr.splits.generate_hash_split(main_split.train_patient_ids, 87, frac_test=0.15)

print(train_split.train_patient_ids)
print(train_split.test_patient_ids)

main_dataset = main_split.split_dataset(dataset, index)
train_dataset = train_split.split_dataset(main_dataset['train'], femr.index.PatientIndex(main_dataset['train'], num_proc=4))

print(train_dataset)

In [ ]:
import femr.models.tokenizer

# First, we need to train a tokenizer

# NOTE: A vocab size of 128 is probably too low for a real model. 128 was chosen to make this tutorial quick to run
tokenizer = femr.models.tokenizer.train_tokenizer(
    main_dataset['train'], vocab_size=128, num_proc=4)

# Save the tokenizer to the same directory as the model
tokenizer.save_pretrained(os.path.join(TARGET_DIR, "clmbr_model"))

In [ ]:
import femr.models.processor
import femr.models.tasks

# Second, we need to create batches. We define the CLMBR task at this time

clmbr_task = femr.models.tasks.CLMBRTask(clmbr_vocab_size=64)

processor = femr.models.processor.FEMRBatchProcessor(tokenizer, clmbr_task)

# We can do this one patient at a time
example_batch = processor.collate([processor.convert_patient(train_dataset['train'][0], tensor_type='pt')])

# But generally we want to convert entire datasets
train_batches = processor.convert_dataset(train_dataset, tokens_per_batch=32, num_proc=4)

# Convert our batches to pytorch tensors
train_batches.set_format("pt")

In [ ]:
import transformers

import femr.models.transformer

# Finally, given the batches, we can train CLMBR.
# We can use huggingface's trainer to do this.

transformer_config = femr.models.config.FEMRTransformerConfig(
    vocab_size=tokenizer.vocab_size, 
    is_hierarchical=tokenizer.is_hierarchical, 
    n_layers=2,
    hidden_size=64, 
    intermediate_size=64*2,
    n_heads=8,
)

config = femr.models.config.FEMRModelConfig.from_transformer_task_configs(transformer_config, clmbr_task.get_task_config())

model = femr.models.transformer.FEMRModel(config)

trainer_config = transformers.TrainingArguments(
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    output_dir='tmp_trainer',
    remove_unused_columns=False,
    num_train_epochs=20,

    eval_steps=20,
    evaluation_strategy="steps",

    logging_steps=20,
    logging_strategy='steps',

    prediction_loss_only=True,
)

trainer = transformers.Trainer(
    model=model,
    data_collator=processor.collate,
    train_dataset=train_batches['train'],
    eval_dataset=train_batches['test'],
    args=trainer_config,
)

trainer.train()

model.save_pretrained(os.path.join(TARGET_DIR, 'clmbr_model'))